In [ ]:
import os

# Creating project structure
dirs = ['papers', 'data']
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'✅ Created: {d}/')

# creat .gitignore
with open('.gitignore', 'w') as f:
    f.write(""".env
__pycache__/
*.pyc
.ipynb_checkpoints/
papers/
data/
*.pdf
""")

# create .env.example
with open('.env.example', 'w') as f:
    f.write("GROQ_API_KEY=your_groq_api_key_here\n")

print(' .gitignore created')
print(' .env.example created')

print('\nProject structure:')
print('paper-chat/')
print('├── papers/     ← For PDF files')
print('├── data/       ← for ChromaDB')
print('└── .env        ← API key')

In [ ]:
import os
import fitz
import chromadb
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sentence_transformers import SentenceTransformer
from typing import List
from dotenv import load_dotenv

load_dotenv()

class VectorStore:
    def __init__(self):
        self.client     = chromadb.Client()
        self.model      = SentenceTransformer('all-MiniLM-L6-v2')
        self.collection = self.client.get_or_create_collection(
            name='papers',
            metadata={'hnsw:space': 'cosine'}
        )

    def add_chunks(self, chunks: List[str], source: str):
        embeddings = self.model.encode(chunks).tolist()
        ids        = [f'{source}_{i}' for i in range(len(chunks))]
        self.collection.add(
            documents=chunks,
            embeddings=embeddings,
            ids=ids,
            metadatas=[{'source': source, 'chunk_id': i} for i in range(len(chunks))]
        )

    def search(self, query: str, n_results: int = 5) -> List[str]:
        embedding = self.model.encode([query]).tolist()
        results   = self.collection.query(
            query_embeddings=embedding,
            n_results=n_results
        )
        return results['documents'][0]

    def clear(self):
        self.client.delete_collection('papers')
        self.collection = self.client.get_or_create_collection('papers')

In [ ]:
from groq import Groq

class RAGPipeline:
    def __init__(self):
        self.client       = Groq(api_key=os.getenv('GROQ_API_KEY'))
        self.vectorstore  = VectorStore()
        self.chat_history = []

    def extract_text(self, pdf_path: str) -> str:
        doc  = fitz.open(pdf_path)
        text = ''
        for page in doc:
            text += page.get_text()
        return text

    def chunk_text(self, text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
        words  = text.split()
        chunks = []
        i = 0
        while i < len(words):
            chunks.append(' '.join(words[i:i + chunk_size]))
            i += chunk_size - overlap
        return chunks

    def index_pdf(self, pdf_path: str) -> int:
        self.vectorstore.clear()
        self.chat_history = []
        text   = self.extract_text(pdf_path)
        chunks = self.chunk_text(text)
        self.vectorstore.add_chunks(chunks, os.path.basename(pdf_path))
        return len(chunks)

    def ask(self, question: str) -> str:
        chunks  = self.vectorstore.search(question, n_results=5)
        context = '\n\n---\n\n'.join(chunks)

        system_prompt = f"""You are an expert research assistant analyzing scientific papers.
Answer questions based ONLY on the provided context from the paper.
If the answer is not in the context, say so clearly.
Be precise and cite relevant parts of the text.

Context:
{context}"""

        self.chat_history.append({'role': 'user', 'content': question})

        response = self.client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            max_tokens=1000,
            messages=[
                {'role': 'system', 'content': system_prompt},
                *self.chat_history
            ]
        )

        answer = response.choices[0].message.content
        self.chat_history.append({'role': 'assistant', 'content': answer})
        return answer

rag = RAGPipeline()
print('RAG pipeline ready')

In [ ]:
# --- style ---
display(HTML("""
<style>
.chat-box {
    height: 400px;
    overflow-y: auto;
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 12px;
    background: #fafafa;
    font-family: sans-serif;
}
.msg-user {
    background: #0084ff;
    color: white;
    padding: 8px 12px;
    border-radius: 12px 12px 2px 12px;
    margin: 6px 0 6px 20%;
    text-align: right;
}
.msg-assistant {
    background: #e9e9eb;
    color: black;
    padding: 8px 12px;
    border-radius: 12px 12px 12px 2px;
    margin: 6px 20% 6px 0;
}
.msg-system {
    color: #888;
    font-size: 0.85em;
    text-align: center;
    margin: 4px 0;
}
</style>
"""))

# --- widgets ---
pdf_input    = widgets.Text(
    placeholder='Path to PDF, e.g. paper.pdf',
    description='PDF path:',
    layout=widgets.Layout(width='400px')
)
index_btn    = widgets.Button(description='Index PDF', button_style='primary')
status_label = widgets.Label(value='No paper loaded')

chat_output  = widgets.Output()
question_input = widgets.Text(
    placeholder='Ask a question about the paper...',
    layout=widgets.Layout(width='500px')
)
ask_btn      = widgets.Button(description='Ask', button_style='success')
clear_btn    = widgets.Button(description='Clear chat', button_style='warning')

chat_html    = ['<div class="chat-box" id="chat">']

def render_chat():
    with chat_output:
        clear_output(wait=True)
        display(HTML(''.join(chat_html) + '</div>'))

def on_index(b):
    path = pdf_input.value.strip()
    if not path or not os.path.exists(path):
        status_label.value = '❌ File not found'
        return
    status_label.value = '⏳ Indexing...'
    try:
        n = rag.index_pdf(path)
        status_label.value = f'✅ Indexed {n} chunks from {os.path.basename(path)}'
        chat_html.clear()
        chat_html.append('<div class="chat-box">')
        chat_html.append(f'<div class="msg-system">📄 Loaded: {os.path.basename(path)}</div>')
        render_chat()
    except Exception as e:
        status_label.value = f'❌ Error: {e}'

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return
    if 'No paper' in status_label.value or '❌' in status_label.value:
        status_label.value = '⚠️ Please index a PDF first'
        return

    question_input.value = ''
    chat_html.append(f'<div class="msg-user">{question}</div>')
    chat_html.append('<div class="msg-system">⏳ Thinking...</div>')
    render_chat()

    try:
        answer = rag.ask(question)
        # remove "thinking"
        chat_html.pop()
        chat_html.append(f'<div class="msg-assistant">{answer}</div>')
    except Exception as e:
        chat_html.pop()
        chat_html.append(f'<div class="msg-system">❌ Error: {e}</div>')

    render_chat()

def on_clear(b):
    rag.chat_history = []
    chat_html.clear()
    chat_html.append('<div class="chat-box">')
    chat_html.append('<div class="msg-system">Chat cleared</div>')
    render_chat()

index_btn.on_click(on_index)
ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

# Enter
def on_enter(event):
    if event['name'] == 'value' and question_input.value.endswith('\n'):
        question_input.value = question_input.value.strip()
        on_ask(None)

# --- layout ---
display(widgets.VBox([
    widgets.HTML('<h2>📄 Paper Chat</h2>'),
    widgets.HBox([pdf_input, index_btn, status_label]),
    widgets.HTML('<hr>'),
    chat_output,
    widgets.HBox([question_input, ask_btn, clear_btn]),
]))

render_chat()